# M5 — App y despliegue (Turno Juan Esteban) — validación en Kaggle
### SI4006 · Proyecto Integrador · Universidad EAFIT

Este notebook levanta `m5/app/app.py` (Gradio) sobre el sistema completo de M1+M2+M3+M4
en una GPU T4 de Kaggle, con link público (`share=True`) para la demo en vivo del
jueves 5 nov.

**Antes de correr, en Kaggle:**
- *Settings → Accelerator* → GPU T4 x2 (o x1, alcanza).
- *Settings → Internet* → On (necesita cuenta verificada por teléfono).
- Este notebook clona la rama `juanes-m5` del repo. **Debe estar pusheada a GitHub**
  antes de correr esto (`git push origin juanes-m5` desde el local), si no, el clone
  falla.


## 1 · Preparar el entorno

In [ ]:
import os

RAMA = "juanes-m5"

if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia

!git log -1 --oneline  # confirmar a simple vista qué commit quedó activo


In [ ]:
%pip install -q -r requirements.txt

# Mismo arreglo que en los notebooks de M2/M3/M4: el entorno trae una version vieja
# de torchao preinstalada que hace tronar a `peft` al importarse.
!pip uninstall -y -q torchao


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Settings -> Accelerator -> GPU T4, "
          "y vuelve a correr esta celda.")


## 2 · Cargar la app y calentar los modelos

`_calentar()` fuerza la carga de los 4 modelos (cerebro, BETO, RAG, `ver()`) una sola
vez, antes de que el primer enfermero haga una consulta real. `health()` todavía no
tiene nada real que reportar -- `m5/observabilidad.py` es de María Alejandra y lo
entrega el lunes 2 nov. Mientras tanto, `app.py` usa un *fallback* que no rompe nada;
en el momento en que ella suba su archivo, esto queda enchufado sin tocar código.

In [ ]:
import sys
sys.path.insert(0, "m5/app")
import app

print("Versión (tag de Git):", app._version_tag())
print("health():", app.health())

app._calentar()


## 3 · Probar el backend (`consultar`) antes de levantar la interfaz

Dos casos ya conocidos de M4, para confirmar que la tarjeta arma bien las tres
señales reales (no los mocks que usamos en local sin GPU):
- **Sin foto**: un síntoma que BETO y el RAG deberían marcar urgente.
- **Con foto**: la imagen `ISIC_0024876.jpg` (nevus benigno confirmado por biopsia) +
  un síntoma vago -- el caso real donde `ver_imagen` y el RAG acertaron en
  'no_urgente' pero BETO forzó 'urgente' y la regla de seguridad escaló el veredicto.
  Debe verse exactamente esa lectura en el bloque de decisión.

In [ ]:
decision, bloque_foto, evidencia = app.consultar(
    "Tengo un lunar que me cambió de color y de forma en los últimos meses.", None,
)
print(decision)
print()
print(bloque_foto)


In [ ]:
_imagen_prueba = "m4/pipeline/eval_tool/imagenes/ISIC_0024876.jpg"
assert os.path.exists(_imagen_prueba), "no se encontró la imagen de prueba del eval_tool"

decision, bloque_foto, evidencia = app.consultar(
    "Le adjunto una foto de la lesión, ¿debería verla pronto?", _imagen_prueba,
)
print(decision)
print()
print(bloque_foto)
print()
print(evidencia)


## 4 · Levantar la interfaz con link público

Esta celda **se queda corriendo** (bloquea) mientras la app esté viva -- es lo que se
usa el jueves en la demo. Para probar hoy:

1. Correr la celda y abrir el link `https://....gradio.live` que imprime.
2. Escribir un síntoma sin foto -> confirmar que el bloque de decisión y el de
   evidencia se ven bien.
3. Subir la foto de `m4/pipeline/eval_tool/imagenes/ISIC_0024876.jpg` (descargarla del
   repo o de este Kaggle) con el mismo síntoma vago -> confirmar que el bloque de foto
   muestra clase/score/calidad y que la decisión final da 'urgente' con la nota de la
   regla de seguridad.
4. Probar una foto borrosa o muy pequeña (cualquier imagen rara) -> confirmar que NO
   dice "no urgente" sino el aviso de no evaluable.
5. Anotar cualquier ajuste de texto/formato que haga falta, para corregir en
   `m5/app/app.py` antes del tag `demo-v1` del martes.

**Al terminar de probar, interrumpir esta celda (botón de stop)** para no seguir
gastando las ~30 h semanales de GPU de la cuenta.

In [ ]:
demo = app.construir_app()
demo.launch(share=True, debug=True)


---
Con esto queda probada la app de punta a punta en una T4 real de Kaggle: carga única
de los 4 modelos, la tarjeta de 3 bloques con casos reales (no mocks), y el link
público que se usará el jueves. Pendiente: enchufar `health()`/`log_consulta()` reales
en cuanto María Alejandra suba `m5/observabilidad.py`, y repetir esta corrida si
Camilo confirma que 4 bits gana (cambiaría qué modelo carga `_calentar()`).